In [1]:
import numpy as np

from genriesz import (
    grr_did,
    SquaredGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
)

rng = np.random.default_rng(0)

In [18]:
def draw_panel(n: int, d_z: int, tau: float, seed: int = 0):
    rng = np.random.default_rng(seed)
    Z = rng.normal(size=(n, d_z))

    logits = 0.6 * Z[:, 0] - 0.25 * Z[:, 1]
    e = 1.0 / (1.0 + np.exp(-logits))
    D = rng.binomial(1, e, size=n).astype(int)

    mu = 0.5 * Z[:, 0] - 0.2 * Z[:, 1] ** 2
    trend = 0.5 + 0.7 * Z[:, 0] + Z[:, 0] * Z[:, 1]**2  # common trend that depends on Z

    u = rng.normal(scale=1.0, size=n)  # unit fixed effect

    Y0 = mu + u + rng.normal(scale=1.0, size=n)
    Y1 = mu + trend + u + tau * D + rng.normal(scale=1.0, size=n)

    X = np.column_stack([D.astype(float), Z])
    return X, Y0, Y1, D

tau_true = 1.0

# Large population for an approximate truth
X_pop, Y0_pop, Y1_pop, D_pop = draw_panel(n=200_000, d_z=5, tau=tau_true, seed=1)

did_without_cov = np.mean((Y1_pop - Y0_pop)[D_pop == 1]) - np.mean((Y1_pop - Y0_pop)[D_pop == 0])  #  DID without covariates
# Our target here is "ATT on ΔY", whose true value equals tau_true by construction.
print("True tau (by construction):", tau_true)
print("Naive DID (difference in mean ΔY):", did_without_cov)

True tau (by construction): 1.0
Naive DID (difference in mean ΔY): 1.8959194276360787


In [19]:
(Y1_pop - Y0_pop)[D_pop == 0]


array([  4.20068093,  -1.13362257,   1.06767099, ...,   1.37468002,
       -15.88947778,   1.94819706], shape=(100266,))

In [20]:
# Sample a dataset from the same DGP
X, Y0, Y1, D = draw_panel(n=6000, d_z=5, tau=tau_true, seed=0)

psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

gen = SquaredGenerator(C=0.0).as_generator()

res = grr_did(
    X=X,
    Y0=Y0,
    Y1=Y1,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res.summary_text())


DID estimates (n=6000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.5468841195092563, max_abs_smd_weighted=0.002480183228107233, ess_treated=3017.9852990018926, ess_control=2036.7126969198098

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.02255     0.0227821         [ 0.977893,  1.0672]           0
RW                 1.01624     0.0894601        [ 0.840902,  1.19158]           0
ARW                1.02803     0.0651263        [ 0.900385,  1.15567]           0
TMLE               1.02793     0.0651536        [ 0.900227,  1.15562]           0


In [21]:
for key, est in res.estimates.items():
    err = est.estimate - tau_true
    print(f"{key:>12s}: estimate={est.estimate: .6f},  error={err: .6f}")


          rw: estimate= 1.016241,  error= 0.016241
          ra: estimate= 1.022545,  error= 0.022545
         arw: estimate= 1.028030,  error= 0.028030
        tmle: estimate= 1.027925,  error= 0.027925


In [22]:
from genriesz import UKLGenerator, BPGenerator

branch = lambda x: int(x[0] == 1.0)

generator_grid = [
    ("SQ", SquaredGenerator(C=0.0).as_generator()),
    ("UKL (C=1)", UKLGenerator(C=1.0, branch_fn=branch).as_generator()),
    ("BP (omega=0.1, C=1)", BPGenerator(C=1.0, omega=0.1, branch_fn=branch).as_generator()),
    ("BP (omega=0.2, C=1)", BPGenerator(C=1.0, omega=0.2, branch_fn=branch).as_generator()),
    ("BP (omega=0.5, C=1)", BPGenerator(C=1.0, omega=0.5, branch_fn=branch).as_generator()),
]

penalty_grid = [
    {"penalty": "l2", "lam": 1e-4},
    {"penalty": "l2", "lam": 1e-3},
    {"penalty": "l1", "lam": 1e-4},
    {"penalty": "lp", "lam": 1e-3},
]

rows = []
for gname, gen_i in generator_grid:
    for cfg in penalty_grid:
        res_i = grr_did(
            X=X,
            Y0=Y0,
            Y1=Y1,
            basis=phi,
            generator=gen_i,
            cross_fit=True,
            folds=3,
            random_state=0,
            estimators=("ra", "rw", "arw", "tmle"),
            outcome_models="shared",
            outcome_link="identity",
            riesz_penalty=cfg["penalty"],
            riesz_lam=cfg["lam"],
            max_iter=250,
            tol=1e-8,
        )

        row = {
            "generator": gname,
            "penalty": cfg["penalty"],
            "lam": cfg["lam"],
        }
        for k in ("ra", "rw", "arw", "tmle"):
            e = res_i.estimates[k]
            row[f"{k}"] = e.estimate
            row[f"{k}_se"] = e.se
            row[f"{k}_err"] = e.estimate - tau_true
        rows.append(row)

import pandas as pd

df = pd.DataFrame(rows)
df = df.sort_values(by="arw_err", key=lambda s: np.abs(s))
display(df)

,generator,penalty,lam,ra,ra_se,ra_err,rw,rw_se,rw_err,arw,arw_se,arw_err,tmle,tmle_se,tmle_err
5,UKL (C=1),l2,0.0010,1.021577,0.022641,0.021577,0.978340,0.089410,-0.021660,1.032509,0.064450,0.032509,1.031317,0.064496,0.031317
7,UKL (C=1),lp,0.0010,1.021577,0.022641,0.021577,0.978340,0.089410,-0.021660,1.032509,0.064450,0.032509,1.031317,0.064496,0.031317
4,UKL (C=1),l2,0.0001,1.021577,0.022641,0.021577,0.975760,0.089505,-0.024240,1.032890,0.064443,0.032890,1.031615,0.064490,0.031615
6,UKL (C=1),l1,0.0001,1.021577,0.022641,0.021577,0.975776,0.089496,-0.024224,1.032897,0.064432,0.032897,1.031622,0.064479,0.031622
9,"BP (omega=0.1, C=1)",l2,0.0010,1.021577,0.022641,0.021577,0.984304,0.088906,-0.015696,1.033462,0.064239,0.033462,1.032307,0.064289,0.032307
11,"BP (omega=0.1, C=1)",lp,0.0010,1.021577,0.022641,0.021577,0.984304,0.088906,-0.015696,1.033462,0.064239,0.033462,1.032307,0.064289,0.032307
10,"BP (omega=0.1, C=1)",l1,0.0001,1.021577,0.022641,0.021577,0.981611,0.088988,-0.018389,1.033885,0.064225,0.033885,1.032641,0.064276,0.032641
8,"BP (omega=0.1, C=1)",l2,0.0001,1.021577,0.022641,0.021577,0.981526,0.088998,-0.018474,1.033910,0.064236,0.033910,1.032663,0.064287,0.032663
15,"BP (omega=0.2, C=1)",lp,0.0010,1.021577,0.022641,0.021577,0.989529,0.088485,-0.010471,1.034499,0.064024,0.034499,1.033365,0.064079,0.033365
13,"BP (omega=0.2, C=1)",l2,0.0010,1.021577,0.022641,0.021577,0.989529,0.088485,-0.010471,1.034499,0.064024,0.034499,1.033365,0.064079,0.033365
